# ✍️ MNIST 서빙 (모델 6개 + 앙상블) — Colab 에서 학습 · 서빙 · 외부 공개

이 노트북 하나로 다음을 합니다.
1. 저장소에서 이 프로젝트 폴더만 받기
2. (선택) GPU 로 다시 학습하고 세 가지 형식으로 저장
3. API 서버 띄우기
4. Cloudflare 터널로 **누구나 접속할 수 있는 https 주소** 만들기

> 학습을 건너뛰어도 저장소에 들어 있는 모델로 바로 서빙됩니다.
> 실행 : 위에서부터 차례로 실행하세요. 런타임 유형은 CPU 도 됩니다 (학습할 때만 GPU 권장).

## 1. 프로젝트 받기

In [ ]:
REPO = "https://github.com/stella1298/AIFFEL_Quest_EPA.git"
SUBDIR = "Serving/Serving02"      # 이 프로젝트가 있는 폴더

!rm -rf /content/repo
!git clone --depth 1 --filter=blob:none --sparse {REPO} /content/repo
!cd /content/repo && git sparse-checkout set {SUBDIR}
%cd /content/repo/{SUBDIR}
!ls

In [ ]:
# 서빙에 필요한 패키지. Colab 에 이미 있는 numpy·pillow 등은 버전만 맞춥니다
!pip install -q -r requirements.txt

## 2. (선택) 다시 학습하기
Colab 에는 torch·torchvision 이 이미 깔려 있으므로 ONNX 내보내기용 패키지만 더 설치합니다.
GPU 런타임이어도 이 스크립트는 CPU 로 학습합니다 (모델 6개, CPU 로 약 6분). 일부만 학습하려면 `--models cnn vit` 처럼 적습니다.

In [ ]:
RETRAIN = False   # True 로 바꾸면 다시 학습합니다

if RETRAIN:
    !pip install -q onnx==1.23.2 onnxscript==0.7.2
    !python -m training.train --epochs 3
!cat models/metrics.json | head -40

## 3. API 서버 띄우기 (백그라운드)

In [ ]:
import os, secrets, subprocess, time, requests

# 공개 주소를 만들 것이므로 기본 키 대신 새 키를 씁니다
ADMIN_KEY = "sk-" + secrets.token_urlsafe(18)
DEMO_KEY = "demo-" + secrets.token_urlsafe(6)
env = {**os.environ, "API_KEYS": f"admin:{ADMIN_KEY},demo:{DEMO_KEY}", "PUBLIC_DEMO_KEY": DEMO_KEY}

server = subprocess.Popen(["python", "-m", "uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"],
                          env=env, stdout=open("server.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        if requests.get("http://127.0.0.1:8000/ready", timeout=1).status_code == 200:
            break
    except requests.RequestException:
        time.sleep(1)
print(requests.get("http://127.0.0.1:8000/ready").json())
print("관리자 키 :", ADMIN_KEY)
print("데모 키   :", DEMO_KEY, "(그림판에 자동으로 채워집니다)")

## 4. 외부 공개 주소 만들기 (Cloudflare Quick Tunnel)
계정 없이 `https://xxxx.trycloudflare.com` 주소가 생깁니다. 이 셀을 멈추거나 런타임을 끄면 주소도 사라집니다.

In [ ]:
import re
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared

tunnel = subprocess.Popen(["./cloudflared", "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8000"],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
PUBLIC_URL = None
for line in tunnel.stdout:
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        PUBLIC_URL = m.group(0)
        break
print("✍️ 그림판   :", PUBLIC_URL)
print("📘 API 문서 :", PUBLIC_URL + "/docs")

## 5. 공개 주소로 직접 호출해 보기

In [ ]:
import base64, io
import numpy as np
from PIL import Image

data = np.load("samples/mnist_test_200.npz")
buf = io.BytesIO(); Image.fromarray(data["images"][1]).save(buf, format="PNG")
img_b64 = "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode()

time.sleep(3)   # 터널이 막 열린 직후에는 몇 초 걸릴 수 있습니다
r = requests.post(f"{PUBLIC_URL}/v1/predict", headers={"X-API-Key": ADMIN_KEY}, json={"image": img_b64})
body = r.json()
print("정답", int(data["labels"][1]), "→ 최종 답", body["ensemble"]["pred"], "· 모델별", {k: v["pred"] for k, v in body["results"].items()})
print("키 없이 :", requests.post(f"{PUBLIC_URL}/v1/predict", json={"image": img_b64}).status_code)

## 6. 끄기

In [ ]:
tunnel.terminate(); server.terminate()
print("터널과 서버를 껐습니다. 공개 주소는 더 이상 열리지 않습니다.")